<div style="
    background-color: #7BAFD4;
    border-radius: 18px;
    padding: 22px 26px;
    display: flex;
    align-items: center;
    gap: 24px;
    margin-bottom: 18px;
">
    <img src="592_avatar_lr.png" alt="GEOG 592"
        style="width:110px;height:110px;border-radius:22px;">
    <div>
        <div style="
            font-family: Impact, 'Arial Black', Arial, sans-serif;
            font-size: 42px;
            font-weight: 900;
            letter-spacing: 2px;
            line-height: 1;
            color: white;
        ">GEOG 592</div>
        <div style="
            font-family: Arial, Helvetica, sans-serif;
            font-size: 22px;
            font-weight: 700;
            color: #13294B;
            margin-top: 8px;
        ">GIS Programming</div>
        <div style="
            font-family: Arial, Helvetica, sans-serif;
            font-size: 16px;
            color: #13294B;
            margin-top: 6px;
        ">Module 3, Day 2: Cleaning, Validation, and Summaries</div>
    </div>
</div>

# Module 3: Day 2
## Cleaning, Validating, and Summarizing Data with pandas

Today we use pandas to turn the Açaí CSV into a dataset that is safer to analyze.

### Today you will learn to:
- convert numeric-looking strings into numeric data
- detect missing values
- create a Boolean data-quality flag
- distinguish `dropna()` from `fillna()`
- inspect unique and duplicate values
- calculate descriptive summaries
- sort and filter using summary information
- export a cleaned CSV

## Import pandas and load the data

Use `read_csv()` to load the data. The `skiprows` parameter allows you to tell Panda where your data actually starts.

It's good to inspect your data with `head()` after it is loaded.

In [ ]:
import pandas as pd

filename = "tabela1613.csv"
df = pd.read_csv(filename, skiprows = 4) 
print(df.columns)
df.head(3)

Let's translate the column names in Pandas (you could also have done this in Excel before loading the file).

use a `print()` statement or `head()` to verify the new column names.


In [ ]:
df.columns = ['MU', 'geoID', 'name', 'empty', 'units', 'acai', 'units2']
print(df['geoID'].iloc[1])
df.head(5)

More cleaning: Let's remove the unnecesary columns. 

In [ ]:
df2 = df[["geoID", "name", "acai", "units"]].copy()
df2.head()

### Why `.copy()`?

The expression `df[["geoID", "name", "acai", "units"]]` returns a filtered DataFrame.

> ```python
> df_filtered = df[["geoID", "name", "acai", "units"]]
> ```
> This example function call creates a variable `df_filtered` that can actually refer to > the same memory as `df`, so *modifying `df2` would also modify your original dataset as a side effect, and possibly produce a warning.

Instead, we use `.copy()` to get an **independent** working object that we can modify without warnings or side-effects.

## Data types
Now let's check the datatypes in the file.

In [ ]:
df2.dtypes

Several columns still look numeric but are stored as strings. A CSV file does not guarantee that values are interpreted with the type we want.

## Convert `Data_Value` to Numeric

`pd.to_numeric()` converts a Series to numbers.

Using `errors="coerce"` means that values that cannot be converted become `NaN` rather than stopping the program.

In [ ]:
df2["acai"] = pd.to_numeric(
    df2["acai"],
    errors="coerce"
)

# df2["acai"].dtype
df2["acai"].head()

### What is `NaN`?

`NaN` is an abbreviation for **Not a Number**. pandas commonly uses it to represent missing numeric data.

Detect Missing Values

`isna()` returns `True` where data are missing.

In [ ]:
df2["acai"].isna().head(10)

Count the missing values by adding the Boolean values. In Python, `True` behaves like 1 and `False` like 0 in this context.

In [ ]:
missing_count = df2["acai"].isna().sum()
print("Missing Data_Value records:", missing_count)

### Missing vs. misinterpreted

We treated all non-numeric data as "missing," but actually there is metadata at the end of the csv explaining the meaning of some values that can't be interpreted as numbers. 
(Commonly, metadata are placed at the top, or in another table, but in this CSV it's at the bottom of the dataset). 


In [ ]:
# Since we made a copy, we have not changed the original data frame 
# so we can read the metadata

df.tail(8)

From the data description we learn that '-' means "Absolute zero, not resulting from a calculation or rounding. Example: In a given municipality, there are no 14-year-old people with no formal education."
and that  the `'...'` means  "Value unavailable. Example: Bean production in a particular municipality was not surveyed, or the municipality did not exist in the year of the survey."

So depending on how we want to display information `'-'` could be 0 and `'...'` could be `NaN`. For the moment we are going to treat them both as `NaN`. 

### Missingness as a data-quality flag

Sometimes it is useful to create a new Boolean column that records whether a value is missing. This is often called a **missing-data indicator** or a **data-quality flag**.

In [ ]:
df2["Data_Value_missing"] = df2["acai"].isna()
df2.head(10)

This preserves information about *which rows* had a problem, rather than only reporting a total count. 

### `dropna()` vs. `fillna()`

There are different ways to handle missing values.

1.  Remove records with missing values

    ```python
    df2.dropna(subset=["acai"])
    ```
2. Replace missing values
    ```python
    df2["acai"].fillna(...)
    ```

Do **not** replace missing values without thinking about what your choice means!
Filling with 0, a mean, or another value changes the data and should be justifiable.

In [ ]:
analysis = df2.dropna(subset=["acai"]).copy()
print("Rows available for analysis:", len(analysis))

## Unique Values

`unique()` shows the distinct values. `nunique()` counts them.

In [ ]:
## check if all the values are stored in the same units
df2["units"].unique()


In [ ]:
print(len(df), 'in the original df')
print(len(df2[df2["units"] == 'Quilogramas por Hectare']), 'number of rows with Quilogramas por Hectare')
print('difference:',len(df) - len(df2[df2["units"] == 'Quilogramas por Hectare']) )

In [ ]:
## checking if we have repeating location values 
df2["geoID"].nunique() 
print('number of repeating values:', len(df2["geoID"]) - df2["geoID"].nunique() ) 

In [ ]:
# Note that the metadata is causing issues
df2["geoID"].tail()

Obviously there is some information in that table that we do not need. Let's find a way to solve this (without using Excel).

In [ ]:
df2["geoID"] = pd.to_numeric(
    df2["geoID"],
    errors="coerce"
).astype("Int64")
missing_count = df2["geoID"].isna().sum()
print('number of NA:', missing_count)

I am getting the idea that we have 22 rows of non-numeric information, but before we delete them I want to show you an example of how `value_counts()` can be used, since it is a very useful method. 
First I am going to create two new columns that will be constructed from the `name` column by extracting the values inside the parenthesis. So for example; Cabixi (RO) will be seperated into Cabixi and RO. 

In [ ]:
df2[["Municipality", "State"]] = df2["name"].str.extract(r"^(.*?)\s*\(([^)]+)\)$") ## I am using regex, and as mentioned in class, I like using AI to create regex these days. 
df2.head()

In [ ]:
## now let's count the number of how many instances we have of each state
df2["State"].value_counts()

From Wikipedia, The Federal Dristict (DF) is Located in the Center-West Region, it is the smallest Brazilian federal unit and the only one that has no municipalities. Also from Wikipedia: Minas Gerais' territory is subdivided into 853 municipalities, the largest number among Brazilian states. 

## Check for Duplicate IDs

A geographic identifier such as `LocationID` should generally identify one county record in this dataset.

In [ ]:
duplicate_ids = df2["geoID"].duplicated().sum()
print("Number of duplicate geoID:", duplicate_ids)

To see the actual duplicate rows, use the Boolean result from `duplicated()` as a filter.

In [ ]:
df2[df2["geoID"].duplicated(keep=False)][
    ["name", "State", "geoID"]]

Let's go ahead and delete rows with NaN in several columns using dropNA.  

In [ ]:
df2.dropna(
    subset=["name", "State", "geoID"], ## this expects the NaN to be in all these columns
    how="all", ## row is removed only when every specified column is missing.
    inplace=True ## modify the existing df2 directly.
)

## check for duplicates once it has been cleaned of NaN NaN NaN
df2[df2["geoID"].duplicated(keep=False)][
    ["name", "State", "geoID"]]

In [ ]:
## Notice that we have changed the shape of df2
print(df2.shape)
print(df.shape)

## Descriptive Statistics

Once `Acai` is numeric, we can calculate summaries directly from the Series object.

In [ ]:
print("Minimum:", df2["acai"].min())
print("Maximum:", df2["acai"].max())
print("Mean:", df2["acai"].mean())
print("Median:", df2["acai"].median())

### Why compare mean and median?

The mean and median describe the center of a distribution in different ways. Their relationship can give us an early clue about whether a distribution is symmetric or skewed.

Tomorrow we will look at the distribution visually.

In [ ]:
df2["acai"].describe()

## Sort the Data

Sort by acai production value (highest to lowest).

In [ ]:
df2.sort_values(
    "acai",
    ascending=False
)[["name", "State", "acai"]].head(10)

### Which municipio has the maximum value?

Instead of scanning the table manually, combine sorting with `iloc`.

In [ ]:
highest = df2.sort_values(
    "acai",
    ascending=False
).iloc[0]

print(highest["name"], highest["acai"])

or use the max value in case you have two counties with similar max values (not the case in this example)

In [ ]:
max_Data_Value = df2['acai'].max()
print(df2['name'][df2['acai']== max_Data_Value] )

### A First Look at `groupby()`

`groupby()` lets us divide rows into groups and calculate a summary for each group.

We can ask for the average `acai` in each state:

In [ ]:
results = df2.groupby("State")["acai"].sum()
print(results.sort_values(ascending=False))

Read the expression from left to right:

1. `df2` is the DataFrame.
2. `.groupby("State")` creates groups based on that column.
3. `["acai"]` selects the numeric variable we want to summarize.
4. `.sum()` calculates the sum for each group.

This is only an introduction to `groupby()`. We will use grouping again later when spatial joins create categories such as points within different polygons.

## Export the Cleaned Data

pandas can write a DataFrame directly to CSV.

In [ ]:
df2.to_csv(
    "cleaned_acai.csv",
    index=False
)

The `index=False` argument prevents pandas from writing its row index as an extra CSV column.

## Some questions: 

1. What does `errors="coerce"` do?
2. What is the difference between detecting missing values and deciding what to do about them?
3. Why might a missing-data flag be useful?
4. Why should you avoid automatically replacing every missing value with 0?
5. How does `selected_columns.to_csv()` demonstrate dot notation?